# Exploración del motor

Este notebook recorre los conceptos del hito 3 de EDsim ejecutando las funciones del paquete
`edsim`: el paciente como proceso de SimPy, el log de eventos como única salida, los recursos
retenidos en tramos distintos, la prioridad con desalojo, los turnos, el calentamiento, los días
punta, la validación sin fórmula cerrada y los números aleatorios comunes.

Toda la lógica vive en el paquete y está testeada; aquí solo se llama a sus funciones, se dibuja
y se explica.

## 0. Cómo usar este notebook

- **Kernel:** en Cursor, pulsa *Select Kernel* (arriba a la derecha) y elige el entorno de
  `.venv` del proyecto. Si no aparece, ejecuta antes `uv sync` en la terminal.
- **Ejecutar:** usa *Restart & Run All* (reiniciar y ejecutar todo). El notebook está pensado
  para funcionar de principio a fin sin depender del orden en que ejecutaste las celdas a mano.
- **Semillas:** cada sección fija su semilla en una constante visible. Con la misma semilla
  el resultado es idéntico; cámbiala para ver otra muestra del azar.
- **Duración:** en modo normal tarda unos 3 minutos; casi todo es la validación de la sección 9,
  el calentamiento de la sección 7 y los pares de la sección 10. Para ir más rápido, baja
  `REPLICAS`, `SEMILLAS_CAL` y `PARES` en la celda de parámetros.
- **Modo rápido:** con la variable de entorno `EDSIM_NB_RAPIDO=1` los días, réplicas y semillas
  se reducen y el notebook tarda segundos. Es lo que hace el CI
  (`EDSIM_NB_RAPIDO=1 uv run pytest --nbmake notebooks/`) para comprobar que se ejecuta sin
  errores; los números de ese modo no sirven para sacar conclusiones.
- **Las gráficas desaparecen tras cada commit:** el hook `nbstripout` de pre-commit borra
  resultados e imágenes antes de guardar el notebook en git. Vuelve a ejecutarlo para verlas.
- **El centro y el perfil** se leen de `config/centro.toml` y `config/llegadas.toml`, los mismos
  que usa la CLI. El notebook se ejecuta desde la carpeta `notebooks/`, de ahí las rutas
  `../config/`.

In [ ]:
import dataclasses
import os
from collections import Counter
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.patches import Patch

from edsim.arrivals.profile import load_profile
from edsim.arrivals.rate import MINUTES_PER_DAY, rate
from edsim.arrivals.report import WEEKDAYS
from edsim.engine.analysis import (
    analyze_run,
    daily_means,
    episode_table,
    hourly_means_by_day,
    paired_waits,
)
from edsim.engine.center import load_center
from edsim.engine.engine import simulate_ed
from edsim.engine.events import MemorySink, to_json
from edsim.engine.invariants import check_invariants
from edsim.engine.report import (
    MUTED_COLOR,
    OBSERVED_COLOR,
    SECOND_COLOR,
    THEORY_COLOR,
    format_engine_validation,
    format_run_summary,
    plot_census,
    plot_hourly_census,
    plot_mg1,
    plot_waits_by_level,
)
from edsim.engine.scenarios import exponential_triage, run_scenario, three_boxes
from edsim.engine.targets import MANCHESTER_MAX_WAIT, daily_waits, manchester_compliance
from edsim.engine.timeline import episode_timeline, gantt
from edsim.engine.validation import (
    census_by_day_and_hour,
    check_infinite_server,
    check_mg1,
    check_more_doctors,
    check_operational_laws,
    check_warmup,
    draws_sample,
    mg1_center,
    offered_load_by_pool,
)
from edsim.seeds import child_streams

# --- Parámetros ------------------------------------------------------------------------
# Con EDSIM_NB_RAPIDO=1 (el CI) todo se reduce para que el notebook tarde segundos.
RAPIDO = os.environ.get("EDSIM_NB_RAPIDO") == "1"

CENTRO_TOML = Path("../config/centro.toml")  # el notebook se ejecuta desde notebooks/
LLEGADAS_TOML = Path("../config/llegadas.toml")
CAL = 1 if RAPIDO else 2  # días de calentamiento de cada ejecución
DIAS = 3 if RAPIDO else 9  # ejecución principal: calentamiento y una semana (secciones 1 a 5)
LUNES = 2 if RAPIDO else 7  # el día que se mira: el lunes 12 de enero en modo normal
DIAS_MES = 9 if RAPIDO else 28  # el mes de las secciones 6, 8 y 9 (lunes 5 a domingo 1)
REPLICAS = 2 if RAPIDO else 20  # réplicas de la M/G/1 y de la Mₜ/G/∞ (sección 9)
DIAS_MG1, DIAS_MTGINF = (3, 3) if RAPIDO else (30, 14)
ESTANCIAS = 1_000 if RAPIDO else 200_000  # muestra de pacientes para m(t) (secciones 6 y 9)
SEMILLAS_CAL, DIAS_CAL = (3, 8) if RAPIDO else (20, 11)  # sección 7
PARES, DIAS_PAR = (2, 3) if RAPIDO else (10, 14)  # sección 10: pares con más médicos

CENTRO = load_center(CENTRO_TOML)
PERFIL = load_profile(LLEGADAS_TOML)
DIA = MINUTES_PER_DAY  # 1440 minutos
H = 60  # minutos por hora

# Colores de engine/report.py: lo simulado en azul, la teoría o la capacidad en naranja,
# una segunda curva en aqua y las referencias en gris. Los niveles llevan los colores de
# Manchester: rojo, naranja, amarillo, verde y azul.
NIVEL_COLOR = {1: "#d03b3b", 2: "#eb6834", 3: "#e3b505", 4: "#1baf7a", 5: "#2a78d6"}


def nombre_dia(d):
    """Nombre corto y fecha del día d de la ejecución (el día 0 es el lunes 5 de enero)."""
    fecha = PERFIL.start_date + timedelta(days=int(d))
    return f"{WEEKDAYS[fecha.weekday()][:3]} {fecha.day}"


plt.rcParams["figure.figsize"] = (9, 4)
print("Modo rápido" if RAPIDO else "Modo normal")

## 1. Introducción: del M/M/1 al servicio de urgencias

En la M/M/1 del hito 1 había **un servidor**, **una cola FIFO** y clientes que llegaban con
**tasa constante** λ. En el hito 2 la tasa pasó a ser λ(t). El motor del hito 3 junta esas
llegadas con un servicio de urgencias:

| M/M/1 | Motor del hito 3 |
| --- | --- |
| Cliente | Paciente: un proceso de SimPy que recorre una máquina de estados |
| Un servidor | Tres pools: enfermería de triaje, boxes y médicos, retenidos en tramos distintos |
| λ constante | λ(t) del hito 2, con el día, la hora y el efecto diario Z |
| Servicio exponencial | Triaje, valoración, pruebas y reevaluación, lognormales |
| FIFO | Triaje FIFO; boxes y médicos por prioridad `(nivel, llegada)`, con desalojo del nivel 1 |
| Capacidad fija | Plantilla por turnos (8, 15 y 22 h) |
| Métricas calculadas en la simulación | Un log de eventos del que se deriva todo |

Abajo, el lunes de la ejecución principal: las llegadas por hora que espera el perfil y el
censo medio de cada hora, derivado del log. El censo sigue a la tasa con retraso: los pacientes
se quedan una hora y media de media.

In [ ]:
SEMILLA = 2026
sink = MemorySink()
resultado = simulate_ed(CENTRO, PERFIL, DIAS, SEMILLA, sink=sink, warmup_days=CAL)
eventos = sink.events
analisis = analyze_run(eventos, CENTRO, DIAS, CAL)
print(
    f"{DIAS} días ({CAL} de calentamiento), semilla {SEMILLA}:"
    f" {len(resultado.ground_truth.episodes)} pacientes y {len(eventos)} eventos"
)

horas = np.arange(24)
llegadas_hora = rate(LUNES * DIA + horas * H + H / 2, PERFIL) * H  # sin Z: lo esperado
censo_hora = hourly_means_by_day(analisis.curves, "census", LUNES * DIA, (LUNES + 1) * DIA)[0]
fig, (ax_l, ax_c) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
ax_l.bar(horas, llegadas_hora, color=THEORY_COLOR, width=0.8)
ax_l.set_ylabel("llegadas esperadas\npor hora")
ax_l.set_title(f"{nombre_dia(LUNES)} de enero: llegadas y censo")
ax_c.bar(horas, censo_hora, color=OBSERVED_COLOR, width=0.8)
ax_c.set_ylabel("censo medio\nde la hora")
ax_c.set_xlabel("hora del día")
ax_c.set_xticks(range(0, 24, 2))
for ax in (ax_l, ax_c):
    ax.grid(axis="y", alpha=0.3)
plt.show()

**Para experimentar**

- ¿Cuántas horas de retraso hay entre el pico de llegadas y el pico del censo? ¿Cuadra con una
  estancia media de unos 90 minutos?
- Cambia `SEMILLA`. ¿Cambia la forma del censo o solo su altura?

> **Lo que debes recordar:** el motor es la M/M/1 con más piezas: λ(t), tres recursos retenidos en tramos distintos, prioridades y turnos; el censo sigue a la tasa con el retraso de la estancia.

## 2. Un paciente paso a paso

Cada paciente es una **función generadora de SimPy**: cada `yield` es una espera (un recurso o
un `timeout`) y cada estado es lo que pasa mientras el proceso está parado en un `yield`. El
código entre dos `yield` se ejecuta en un instante, y ahí se emite el evento de la transición,
con `sim_time = env.now`.

Todo lo aleatorio del paciente se sortea **al llegar**, en orden fijo: nivel, duraciones, si
tendrá pruebas y su respuesta, y destino. Lo que no se sortea son las **esperas**: dependen de
los demás pacientes y de la plantilla.

Abajo, un paciente de la ejecución principal con pruebas y alguna espera: sus eventos tal como
están en el log (JSONL) y su recorrido por estados, que `episode_timeline` reconstruye solo con
esos eventos.

In [ ]:
verdad = {e.episode_id: e for e in resultado.ground_truth.episodes}
candidatos = [
    e
    for e in analisis.table
    if e.departure is not None
    and e.arrival >= CAL * DIA
    and verdad[e.episode_id].needs_tests
    and e.box_wait + e.doctor_wait > 0
]
EPISODIO = candidatos[0].episode_id
suyos = [e for e in eventos if e.episode_id == EPISODIO]
print(f"{EPISODIO}: {len(suyos)} eventos. Los tres primeros, tal como están en el log:\n")
for e in suyos[:3]:
    print(to_json(e))
print()
for e in suyos:
    payload = ", ".join(f"{k}={v}" for k, v in e.payload.items())
    print(f"{e.sim_time:10.2f}  {e.event_time[11:19]}  {e.event_type:<20} {payload}")
print(f"\nSorteado al llegar (ground truth): {verdad[EPISODIO]}")

In [ ]:
ESTADOS = {
    "waiting_triage": ("espera de triaje", MUTED_COLOR),
    "triage": ("triaje", "#7b5ea7"),
    "waiting_box": ("espera de box", THEORY_COLOR),
    "waiting_doctor": ("espera de médico", "#e3b505"),
    "assessment": ("valoración", OBSERVED_COLOR),
    "tests": ("pruebas", SECOND_COLOR),
    "reassessment": ("reevaluación", "#1d4e89"),
}
tramos = episode_timeline(eventos, EPISODIO, DIAS * DIA)
inicio = tramos[0].start
fig, ax = plt.subplots(figsize=(11, 2.6))
for s in tramos:
    etiqueta, color = ESTADOS[s.state]
    ax.barh(0, s.end - s.start, left=s.start - inicio, color=color, edgecolor="white", height=0.6)
    if s.end - s.start > 6:
        ax.text((s.start + s.end) / 2 - inicio, 0, etiqueta, ha="center", va="center", fontsize=8)
for e in suyos:
    ax.plot(e.sim_time - inicio, 0.45, "v", color="k", ms=4)
ax.set_yticks([])
ax.set_ylim(-0.45, 0.6)
ax.set_xlim(-3, tramos[-1].end - inicio + 3)
ax.set_xlabel(f"minutos desde la llegada ({EPISODIO})")
ax.set_title("Recorrido del paciente: cada triángulo es un evento del log")
ax.legend(
    handles=[Patch(color=c, label=n) for n, c in ESTADOS.values()],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.45),
    ncol=7,
    fontsize=8,
)
plt.show()
for s in tramos:
    print(f"{ESTADOS[s.state][0]:<18}{s.end - s.start:7.1f} min")

**Para experimentar**

- Elige otro episodio de `candidatos`. ¿Cuál es el estado más largo de su estancia? ¿Es una
  espera o una actividad?
- Busca un paciente sin pruebas: ¿cuántos eventos tiene? ¿Qué estados le faltan?

> **Lo que debes recordar:** el paciente es un proceso que espera en cada `yield` y emite un evento en cada transición; sus sorteos se hacen al llegar, y lo único que depende de los demás son las esperas.

## 3. El log como fuente de verdad

El motor no calcula métricas: **escribe hechos**. El estado del centro en cualquier instante se
obtiene **plegando** los eventos hasta ese instante (*event sourcing*):

- `episode_table`: una fila por episodio, con sus instantes y esperas.
- `step_curves`: censo, colas, ocupados y plantilla vigente, como escaleras que solo cambian en
  los eventos.
- `window_metrics` y `little_laws`: las métricas en la ventana [calentamiento, horizonte].

Como el log es la única fuente, también se puede preguntar **qué se sabía** a una hora
concreta: basta con plegar solo el prefijo del log.

In [ ]:
violaciones = check_invariants(
    eventos, CENTRO, ground_truth=resultado.ground_truth, inside=resultado.inside
)
print(format_run_summary(analisis, violaciones))

In [ ]:
t = LUNES * DIA + 12 * H  # el lunes a las 12:00
prefijo = [e for e in eventos if e.sim_time <= t]
sabido = episode_table(prefijo)
dentro = [e for e in sabido if e.departure is None]
print(f"{nombre_dia(LUNES)} a las 12:00, con los {len(prefijo)} primeros eventos del log:")
print(f"  censo {len(dentro)} (escalera: {analisis.curves.value('census', t)})")
print(f"  esperando triaje {sum(e.triage_start is None for e in dentro)}")
print(
    "  esperando box "
    f"{sum(e.triage_end is not None and e.first_box is None for e in dentro)}"
    f" (escalera: {analisis.curves.value('waiting_box', t)})"
)
print(
    f"  médicos ocupados {analisis.curves.value('doctors_busy', t)}"
    f" de {analisis.curves.value('doctors_staff', t)}"
)
display(plot_census(analisis, CENTRO, PERFIL.start_date))

**Para experimentar**

- Cambia `t` a las 4:00 del lunes. ¿Cuántos pacientes hay? ¿Y médicos de plantilla?
- ¿Por qué el censo puede pasar de 30 si solo hay 30 boxes? ¿Dónde están esos pacientes?

> **Lo que debes recordar:** el log es la única salida del motor; censo, ocupación, esperas y "lo que se sabía" en cada instante se obtienen plegando sus eventos, y por eso cualquier métrica se puede recalcular desde el fichero.

## 4. Recursos retenidos en tramos distintos

Un paciente **retiene el box toda la estancia**, desde `box_assigned` hasta la salida, también
mientras espera los resultados de las pruebas. Al **médico** solo lo retiene durante cada
valoración. Por eso el box suele ser el cuello de botella aunque haya médicos libres.

El diagrama de Gantt de una mañana lo muestra: arriba, cada fila es un box y cada barra un
paciente; abajo, las valoraciones repartidas en **carriles de médico**. Los pools son anónimos,
así que el log no dice qué médico hizo cada valoración: `gantt` las coloca en el mínimo de
carriles sin solapes, que es justo el máximo de médicos ocupados a la vez.

In [ ]:
inicio, fin = LUNES * DIA + 8 * H, LUNES * DIA + 15 * H
g = gantt(eventos, inicio, fin)


def dibuja_gantt(g, ax_box, ax_med, n_boxes, origen):
    for box, barras in g.boxes.items():
        for b in barras:
            ax_box.barh(
                box, b.end - b.start, left=b.start - origen, color=NIVEL_COLOR[b.level], height=0.8
            )
    for carril, barras in enumerate(g.doctors, start=1):
        for b in barras:
            ax_med.barh(
                carril,
                b.end - b.start,
                left=b.start - origen,
                color=NIVEL_COLOR[b.level],
                height=0.8,
                hatch="///" if b.interrupted else None,
                edgecolor="white" if not b.interrupted else "k",
            )
    ax_box.set_ylim(0.3, n_boxes + 0.7)
    ax_box.invert_yaxis()
    ax_med.invert_yaxis()
    ax_box.set_ylabel("box")
    ax_med.set_ylabel("carril de médico")


ocupacion_box = sum(b.end - b.start for bs in g.boxes.values() for b in bs) / (fin - inicio)
ocupacion_med = sum(b.end - b.start for bs in g.doctors for b in bs) / (fin - inicio)
print(
    f"{nombre_dia(LUNES)} de 8 a 15 h: {ocupacion_box:.1f} boxes y {ocupacion_med:.1f} médicos"
    f" ocupados de media; {len(g.doctors)} carriles de médico (12 de plantilla)"
)

fig, (ax_b, ax_m) = plt.subplots(
    2, 1, sharex=True, figsize=(12, 8), gridspec_kw={"height_ratios": [30, max(len(g.doctors), 1)]}
)
dibuja_gantt(g, ax_b, ax_m, CENTRO.boxes, LUNES * DIA)
ax_m.set_xlabel("hora del día")
ax_m.set_xticks(np.arange(8, 16) * H, [f"{h} h" for h in range(8, 16)])
ax_b.set_title(f"{nombre_dia(LUNES)}, de 8 a 15 h: boxes (arriba) y médicos (abajo), por nivel")
fig.legend(
    handles=[Patch(color=c, label=f"nivel {n}") for n, c in NIVEL_COLOR.items()],
    loc="upper right",
    ncol=5,
    fontsize=8,
)
plt.show()

Las barras de los boxes son largas y casi no dejan huecos; las de los médicos son cortas y con
huecos entre ellas. El hueco de un box mientras el paciente espera sus pruebas no lo puede
aprovechar nadie: es tiempo de box sin médico.

**Para experimentar**

- Cambia `inicio` y `fin` a la mañana del domingo anterior (día `LUNES - 1`): ¿quedan boxes
  vacíos? ¿Cuántos carriles de médico hacen falta?
- ¿Qué paciente ocupa más tiempo un box en la mañana? ¿Tenía pruebas?

> **Lo que debes recordar:** el box se retiene toda la estancia y el médico solo en las valoraciones; por eso los boxes se llenan antes que los médicos, y el Gantt reparte las valoraciones en el mínimo de carriles, que coincide con el máximo de médicos ocupados.

## 5. Prioridad y desalojo

Boxes y médicos se ordenan por la tupla **`(nivel, llegada)`**: primero el nivel y, dentro del
nivel, quien llegó antes, pida el recurso cuando lo pida (una reevaluación o la vuelta tras un
desalojo). El triaje es FIFO, porque antes del triaje no se conoce el nivel.

Un **nivel 1** que no encuentra box **desaloja** al paciente de clave mayor (si es de nivel
mayor que 1): el desalojado recibe `simpy.Interrupt`, vuelve a la cola de box con su misma clave
y, al recuperarlo, **reanuda** con el tiempo que le faltaba.

El escenario de tres boxes de la sesión conceptual lo muestra con instantes calculados a mano
(`edsim.engine.scenarios.three_boxes`):

- P1 (nivel 3), P2 (nivel 4) y P3 (nivel 5) ocupan los tres boxes; P4 (nivel 2) espera.
- P5 (nivel 1) termina el triaje en el minuto 20 y desaloja a P3, que llevaba 13 de sus 40
  minutos de valoración.
- P5 sale en 50 y su box va a P4, no a P3, que llegó antes: P4 tiene nivel 2.
- P1 sale en 65; P3 recupera un box y termina sus 27 minutos en 92.

In [ ]:
escenario = three_boxes(CENTRO)
log_escenario = run_scenario(escenario).events
for e in log_escenario:
    if e.event_type in ("box_assigned", "patient_displaced", "patient_departed"):
        print(f"{e.sim_time:5.0f}  {e.episode_id}  {e.event_type:<18} {dict(e.payload)}")

g3 = gantt(log_escenario, 0.0, 100.0)
fig, (ax_b, ax_m) = plt.subplots(
    2, 1, sharex=True, figsize=(10, 4), gridspec_kw={"height_ratios": [3, len(g3.doctors)]}
)
dibuja_gantt(g3, ax_b, ax_m, 3, 0.0)
for box, barras in g3.boxes.items():
    for b in barras:
        ax_b.text(
            (b.start + b.end) / 2,
            box,
            f"P{int(b.episode_id[-1]) + 1}",
            ha="center",
            va="center",
            color="white",
            fontsize=9,
            fontweight="bold",
        )
ax_b.set_yticks([1, 2, 3])
ax_m.set_xlabel("minuto")
ax_b.set_title(
    "Tres boxes: P5 (nivel 1) desaloja a P3 en el minuto 20 (rayado: valoración cortada)"
)
plt.show()

En una semana del centro real la prioridad **no reduce la espera: la reparte**. El nivel 1 casi
no espera box y los niveles 4 y 5 cargan con la espera (con FIFO, la tarea 3.4 midió unos 3,5
minutos de espera de box para todos los niveles).

In [ ]:
display(plot_waits_by_level(analisis.metrics))
desalojos = sum(e.event_type == "patient_displaced" for e in eventos)
print(f"Desalojos en los {DIAS} días: {desalojos}")

**Para experimentar**

- En el escenario, ¿qué pasaría si P5 fuera de nivel 2? Construye otro `Scenario` con
  `hand_draws` (de `edsim.engine.scenarios`) y compáralo.
- ¿Por qué el nivel 1 tiene una espera puerta-médico de varios minutos aunque no espere box?
  (Pista: mira el triaje.)

> **Lo que debes recordar:** la prioridad es la tupla `(nivel, llegada)`, así que el orden no depende de cuándo se pide el recurso; el nivel 1 desaloja con reanudación del tiempo pendiente, y la prioridad no reduce la espera media: la reparte entre niveles.

## 6. Carga ofrecida y turnos

La **carga ofrecida** es el número de recursos que la demanda tendría ocupados **sin colas**:

> m(t) = ∫ λ(s) · P(ocupa el recurso a la edad t − s) ds

Para el censo, "ocupa" es "sigue dentro"; para los boxes, desde el fin del triaje hasta la
salida; para los médicos, durante la valoración y la reevaluación. `offered_load_by_pool` la
calcula con la λ(t) del perfil y una muestra de pacientes del mismo muestreador del motor. Si
m(t) se acerca a la capacidad, las esperas se disparan.

Abajo, el lunes: m(t) frente a la plantilla vigente, que sale del log (`staffing_changed`), y
los ocupados de verdad de la ejecución principal.

In [ ]:
SEMILLA = 61
muestra = draws_sample(CENTRO, ESTANCIAS, SEMILLA)
carga = offered_load_by_pool(PERFIL, muestra, DIAS * DIA)
minutos = np.arange(LUNES * DIA, (LUNES + 1) * DIA)
curvas = analisis.curves
paneles = (
    ("boxes", "boxes_busy", None, "Boxes"),
    ("medicos", "doctors_busy", "doctors_staff", "Médicos"),
    ("enfermeria_triaje", "nurses_busy", "nurses_staff", "Enfermería de triaje"),
)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (pool, ocupados, plantilla, titulo) in zip(axes, paneles, strict=True):
    ax.plot(
        (minutos - LUNES * DIA) / H,
        carga[pool][minutos],
        color=THEORY_COLOR,
        lw=2,
        label="m(t), sin colas",
    )
    real = [curvas.value(ocupados, t) for t in minutos]
    ax.plot(
        (minutos - LUNES * DIA) / H,
        real,
        color=OBSERVED_COLOR,
        lw=1,
        alpha=0.8,
        label="ocupados (log)",
    )
    cap = (
        [CENTRO.boxes] * len(minutos)
        if plantilla is None
        else [curvas.value(plantilla, t) for t in minutos]
    )
    ax.plot((minutos - LUNES * DIA) / H, cap, color="k", ls="--", lw=1.5, label="capacidad")
    ax.set_title(titulo)
    ax.set_xlabel("hora del día")
    ax.set_xticks(range(0, 25, 4))
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle(f"{nombre_dia(LUNES)}: carga ofrecida frente a la plantilla")
plt.tight_layout()
plt.show()
for pool, _, _, titulo in paneles:
    m = carga[pool][minutos]
    print(f"{titulo:<22} m(t) máxima {m.max():5.1f}, media {m.mean():5.1f}")

A mediodía del lunes la carga de los boxes roza los 30 y la de los médicos pasa de 12: la
plantilla está justa. La noche empieza con 7 médicos y una carga de unos 9 que tarda en bajar:
los turnos fijos no siguen a la demanda (la tarea 3.5 midió una espera de médico 15 veces mayor
de 22 a 23 h).

**¿Qué pasa si se quita un médico por la mañana?** El centro tiene la misma plantilla todos los
días, así que se pasa el turno de mañana de 12 a 11 médicos **todos los días** y se comparan,
con la misma semilla, los mismos pacientes: cada uno llega a la misma hora y con los mismos
sorteos, y solo cambian sus esperas (sección 10).

In [ ]:
SEMILLA_MES = child_streams(2026, 5)[2]  # la ejecución larga de validate_engine con semilla 2026
menos = dataclasses.replace(CENTRO, doctors=(CENTRO.doctors[0] - 1, *CENTRO.doctors[1:]))


def ejecuta(centro, dias, semilla):
    """Simula en memoria y devuelve los eventos y el análisis."""
    s = MemorySink()
    simulate_ed(centro, PERFIL, dias, semilla, sink=s, warmup_days=CAL)
    return s.events, analyze_run(s.events, centro, dias, CAL)


eventos_mes, mes = ejecuta(CENTRO, DIAS_MES, SEMILLA_MES)
_, mes_menos = ejecuta(menos, DIAS_MES, SEMILLA_MES)
par = paired_waits(mes.table, mes_menos.table, "door_to_doctor", CAL * DIA, DIAS_MES * DIA)
dia_par = np.floor(par.arrivals / DIA).astype(int)
hora_par = (par.arrivals % DIA) / H
manana = (hora_par >= 8) & (hora_par < 15)
lunes = manana & (dia_par == LUNES)
print(
    f"Médicos por turno: {CENTRO.doctors} frente a {menos.doctors};"
    f" {len(par.episode_ids)} pacientes emparejados"
)
print(
    f"{nombre_dia(LUNES)} de 8 a 15 h: {lunes.sum()} pacientes; puerta-médico medio"
    f" {par.base[lunes].mean():.1f} → {par.other[lunes].mean():.1f} min"
    f" (diferencia media {par.difference[lunes].mean():+.1f})"
)

In [ ]:
semana = np.array(dia_par % 7)  # el día 0 es lunes
dif_dia = [par.difference[manana & (semana == k)] for k in range(7)]
medias = [d.mean() if d.size else np.nan for d in dif_dia]
fig, (ax_l, ax_s) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1, 1.3]})
ax_l.scatter(par.base[lunes], par.other[lunes], s=12, color=OBSERVED_COLOR, alpha=0.7)
tope = max(par.other[lunes].max(initial=1), par.base[lunes].max(initial=1))
ax_l.plot([0, tope], [0, tope], color="k", lw=1, ls=":")
ax_l.set_xlabel("puerta-médico con 12 médicos (min)")
ax_l.set_ylabel("con 11 médicos (min)")
ax_l.set_title(f"{nombre_dia(LUNES)}, 8-15 h: cada punto, un paciente")
ax_s.bar(range(7), medias, color=[THEORY_COLOR if k == 0 else OBSERVED_COLOR for k in range(7)])
for k, d in enumerate(dif_dia):
    ax_s.text(k, (medias[k] if d.size else 0) + 0.3, f"n={d.size}", ha="center", fontsize=8)
ax_s.set_xticks(range(7), [w[:3] for w in WEEKDAYS])
ax_s.set_ylabel("diferencia media del puerta-médico (min)")
ax_s.set_title(f"Un médico menos por la mañana, en {DIAS_MES - CAL} días: cada día de la semana")
ax_s.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
for k in range(7):
    print(f"{WEEKDAYS[k]:>9}: {medias[k]:+5.1f} min ({dif_dia[k].size} pacientes)")

**El mismo médico de menos no pesa lo mismo todos los días.** El fin de semana, con menos
llegadas, la plantilla de 11 sigue holgada y la espera apenas cambia. El lunes, el día más
cargado, la mañana ya está cerca de la saturación y el mismo médico de menos multiplica la
espera. Es la lección de la M/M/1 del hito 1: la espera crece como ρ/(1 − ρ), así que **un mismo
cambio de capacidad importa poco lejos de la saturación y muchísimo cerca de ella**. Por eso la
plantilla se dimensiona con el día punta, no con el día medio, y una ocupación media no basta
para decidir.

**Para experimentar**

- Quita el médico del turno de tarde (`doctors=(12, 11, 7)`). ¿Qué día de la semana sufre más?
- ¿Y si en vez de quitar un médico se quitan dos boxes (`boxes=28`)? ¿Qué espera crece más, la
  de box o la de médico?

> **Lo que debes recordar:** la carga ofrecida m(t) dice cuántos recursos tendría ocupados la demanda sin colas; cuando se acerca a la plantilla, las esperas crecen de forma no lineal: el mismo médico de menos casi no se nota el fin de semana y pesa mucho el lunes.

## 7. Calentamiento

Con λ(t) variable **no hay estado estacionario**: hay un **régimen periódico**, que se repite
cada día y cada semana. La simulación empieza vacía el día 0 a las 0:00, cuando un día real ya
tendría unos 20 pacientes dentro. El **calentamiento** descarta los primeros días completos para
no medir ese arranque falso.

¿Cuántos días hacen falta? Se simulan muchas semillas desde vacío y, para cada hora, se compara
el censo del día d con el del **mismo día de la semana siguiente** (d + 7), emparejado por
semilla, con un IC al 99 % y Bonferroni entre las 24 horas (`check_warmup`). Compararlo con la
media de la semana mezclaría el arranque con el efecto del día de la semana: un lunes siempre
tiene más gente.

In [ ]:
SEMILLA = 35
censo = census_by_day_and_hour(CENTRO, PERFIL, DIAS_CAL, SEMILLA, SEMILLAS_CAL)
dias_cal = range(DIAS_CAL - 7)
fig, axes = plt.subplots(
    1, len(dias_cal), figsize=(3.5 * len(dias_cal) + 1, 3.6), sharey=True, squeeze=False
)
for ax, d in zip(axes[0], dias_cal, strict=True):
    resultado_cal = check_warmup(censo, d, d + 7)
    fuera = [int(x.name[:2]) for x in resultado_cal.details if not x.passed]
    ax.plot(horas, censo[:, d + 7].mean(axis=0), color=THEORY_COLOR, lw=2, label="día d + 7")
    ax.plot(horas, censo[:, d].mean(axis=0), "o-", color=OBSERVED_COLOR, ms=3, label="día d")
    ax.plot(
        fuera,
        censo[:, d, fuera].mean(axis=0),
        "o",
        color="#d03b3b",
        ms=8,
        mfc="none",
        mew=2,
        label="hora fuera del IC",
    )
    ax.set_title(f"día {d} ({nombre_dia(d)}): {len(fuera)} horas fuera")
    ax.set_xlabel("hora")
    ax.set_xticks(range(0, 24, 6))
    ax.grid(alpha=0.3)
    print(f"día {d} frente a {d + 7}: horas fuera {fuera or 'ninguna'}")
axes[0][0].set_ylabel(f"censo medio ({SEMILLAS_CAL} semillas)")
axes[0][0].legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"0 h del día 0: censo {censo[:, 0, 0].mean():.1f}; del día 7: {censo[:, 7, 0].mean():.1f}")

El arranque vacío solo se nota en las primeras horas del día 0: a media mañana del lunes las
llegadas ya han llenado el centro, porque la estancia media es de hora y media. Dos días de
calentamiento (el valor por defecto) dejan mucho margen.

**Para experimentar**

- Con `SEMILLAS_CAL = 5`, ¿sigue saliendo fuera la hora 0 del día 0? ¿Por qué un IC más ancho
  ve menos?
- Si la estancia media fuera de 8 horas (un centro con *boarding*, hito 5), ¿bastarían dos días?

> **Lo que debes recordar:** con λ(t) no hay estado estacionario sino un régimen periódico; el calentamiento se mide en días completos comparando cada día con el mismo día de la semana siguiente, y aquí el arranque vacío solo dura unas horas.

## 8. Días punta

La ocupación media de los boxes en el mes es de un 60–70 %. Parece holgada, pero **una media
esconde los días malos**: cerca de la saturación las esperas crecen de forma no lineal (sección
6), y unos pocos días concentran casi toda la espera de los niveles bajos. Es el hallazgo de la
tarea 3.7, con la misma ejecución larga de `validate_engine`.

Por eso el diseño pide mirar el **porcentaje de días fuera de objetivo**. El objetivo son los
tiempos máximos hasta el médico del sistema Manchester (`MANCHESTER_MAX_WAIT`): inmediato, 10,
60, 120 y 240 minutos por nivel. Un día está **fuera** para un nivel si menos del 90 % de sus
pacientes de ese día llegan al médico dentro del máximo (`manchester_compliance`).

In [ ]:
espera_box = daily_waits(mes.table, "box_wait", CAL, DIAS_MES)
puerta = daily_waits(mes.table, "door_to_doctor", CAL, DIAS_MES)
cumple = manchester_compliance(mes.table, CAL, DIAS_MES, measure="door_to_doctor")  # reloj A
ocupacion_dia = daily_means(mes.curves, "boxes_busy", CAL, DIAS_MES) / CENTRO.boxes
ocupacion_media = mes.metrics.time_averages["boxes_busy"] / CENTRO.boxes

fig, (ax_e, ax_o) = plt.subplots(
    2, 1, sharex=True, figsize=(12, 6.5), gridspec_kw={"height_ratios": [2, 1]}
)
for nivel in (2, 3, 4, 5):
    ax_e.plot(
        espera_box.days,
        espera_box.mean[:, nivel - 1],
        "o-",
        ms=4,
        color=NIVEL_COLOR[nivel],
        label=f"nivel {nivel}",
    )
ax_e.set_ylabel("espera media de box (min)")
ax_e.set_title(f"{DIAS_MES - CAL} días: espera de box por día de llegada y nivel")
ax_e.legend(fontsize=8)
ax_e.grid(alpha=0.3)
ax_o.bar(espera_box.days, 100 * ocupacion_dia, color=OBSERVED_COLOR, width=0.8)
ax_o.axhline(
    100 * ocupacion_media,
    color=THEORY_COLOR,
    lw=2,
    label=f"media del periodo: {100 * ocupacion_media:.0f} %",
)
ax_o.set_ylabel("ocupación de\nboxes del día (%)")
ax_o.set_ylim(0, 100)
ax_o.legend(fontsize=8, loc="lower right")
ax_o.set_xticks(espera_box.days, [nombre_dia(d) for d in espera_box.days], rotation=90, fontsize=8)
plt.tight_layout()
plt.show()

peores = np.argsort(-np.nan_to_num(espera_box.mean[:, 4]))[:3]
for i in peores:
    d = espera_box.days[i]
    print(
        f"{nombre_dia(d)} de enero: espera de box nivel 4 {espera_box.mean[i, 3]:.0f} min,"
        f" nivel 5 {espera_box.mean[i, 4]:.0f} min; ocupación del día {ocupacion_dia[i]:.0%}"
    )

In [ ]:
print("Nivel  máximo  días con pacientes  días fuera  % fuera  puerta-médico medio")
for k in range(5):
    print(
        f"{k + 1:>5}  {MANCHESTER_MAX_WAIT[k]:>6.0f}  {cumple.days_with_patients[k]:>18}"
        f"  {cumple.out[:, k].sum():>10}  {100 * cumple.share_out[k]:>6.0f} %"
        f"  {np.nanmean(puerta.mean[:, k]):>8.1f} min"
    )

fig, ax = plt.subplots(figsize=(12, 2.8))
celdas = np.where(cumple.n > 0, cumple.within, np.nan).T
im = ax.imshow(100 * celdas, aspect="auto", cmap="RdYlGn", vmin=50, vmax=100)
for k in range(5):
    for i in range(len(cumple.days)):
        if cumple.out[i, k]:
            ax.text(i, k, "×", ha="center", va="center", fontsize=8)
ax.set_yticks(range(5), [f"nivel {k + 1}" for k in range(5)])
ax.set_xticks(
    range(len(cumple.days)), [nombre_dia(d) for d in cumple.days], rotation=90, fontsize=8
)
ax.set_title("% de pacientes que llegan al médico dentro del máximo de Manchester (× = día fuera)")
fig.colorbar(im, ax=ax, label="%")
plt.show()

Tres lecturas. Los días y porcentajes que cita el texto son los de la **ejecución normal**
(un mes de 28 días); en modo rápido el mes dura 9 días y no coinciden con la gráfica.

- **Los niveles 3 a 5 esperan pocos minutos la mayoría de los días**, pero en unos pocos días
  punta (los lunes 12 y 26, el miércoles 14, el martes 20 y el jueves 22) el
  nivel 5 espera horas: el nivel 4 queda fuera 3 de 26 días y el 5, 8. La ocupación media del
  periodo es del 64 %, y la de esos días no es mucho mayor (del 69 al 80 %): cerca de la
  saturación, unos puntos más de ocupación disparan la espera. La media del mes no lo enseña;
  el porcentaje de días fuera sí.
- **El nivel 1 está fuera todos los días** con un máximo "inmediato": el triaje es FIFO y dura
  unos 6 minutos (hallazgo de la tarea 3.6). En la realidad un crítico entra directo; el hito 4
  lo decidirá.
- **El nivel 2 también sale fuera casi todos los días**: 10 minutos dan para el triaje y poco
  más. Es un hallazgo nuevo para la calibración del hito 4.

**Para experimentar**

- Cambia `target=0.8` en `manchester_compliance`. ¿Qué niveles cambian de veredicto?
- Añade 2 boxes (`dataclasses.replace(CENTRO, boxes=32)`) y repite: ¿desaparecen los días punta
  de los niveles 4 y 5?

> **Lo que debes recordar:** una ocupación media del 60–70 % puede esconder días saturados; para calibrar hay que mirar los percentiles y el porcentaje de días fuera de los máximos de Manchester por nivel, no solo las medias.

## 9. Validación

Un servicio con λ(t), tres pools, prioridades y desalojo no tiene fórmula cerrada. Se valida con
varios tipos de comprobación, cada uno ciego a algo:

| Nivel | Qué detecta |
| --- | --- |
| 1. Invariantes | Errores de lógica en cualquier ejecución: conservación, capacidad, tiempo, transiciones, nadie se cuela, desalojo |
| 2. Leyes operacionales | Little (L = λ·W) y área de ocupados = suma de atenciones: dos caminos que deben coincidir |
| 3a. M/G/1 | El triaje con una enfermera y λ constante, frente a Pollaczek–Khinchine |
| 3b. Mₜ/G/∞ | Con capacidad infinita, el censo por hora frente a m(t) |
| 4. Condiciones extremas | Capacidad infinita: esperas 0 y estancias exactas |

Las invariantes solo ven la **forma** del log. Un log roto a propósito (aquí, sin un
`box_assigned`) las hace saltar; una duración mal calculada, no.

In [ ]:
quitado = next(i for i, e in enumerate(eventos) if e.event_type == "box_assigned")
roto = eventos[:quitado] + eventos[quitado + 1 :]
violaciones_rotas = check_invariants(roto, CENTRO)
print(f"Log correcto: {len(violaciones)} violaciones")
print(f"Sin el box_assigned de {eventos[quitado].episode_id}: {len(violaciones_rotas)} violaciones")
por_comprobacion = Counter(v.check for v in violaciones_rotas)
for check, n in por_comprobacion.most_common():
    ejemplo = next(v for v in violaciones_rotas if v.check == check)
    print(f"    {check}: {n}  (p. ej. {ejemplo.message})")

**Por qué un solo evento quitado da tantas violaciones.** En la ejecución normal, quitar el
primer `box_assigned` (el de EP-000001, nivel 3, en el minuto 18) da 2.515 violaciones de solo
tres comprobaciones:

- **`transitions` y `capacity`, una cada una.** EP-000001 empieza su valoración sin haber
  entrado en un box: el autómata no admite esa transición y la capacidad ve una valoración en el
  box 1 de un paciente que, según el log, no ocupa ningún box. Tras la primera transición
  inválida, el episodio deja de revisarse: un error da una violación, no una cascada.
- **`nobody_jumps_the_queue`, 2.513.** Para el log, EP-000001 sigue esperando box el resto de
  la ejecución, con su clave (3, 10,5). Cada box que se asigna después a un paciente de clave
  mayor (los niveles 3 que llegaron más tarde y los 4 y 5) cuenta como colarse: 2.510
  episodios, y 3 de ellos dos veces, porque vuelven a recibir box tras un desalojo.

Un hecho que falta no se queda en su episodio: contamina todo lo que se compara con él. Para
diagnosticar, importa más **qué comprobación** salta, y dónde salta primero, que el total. En
modo rápido la ejecución es más corta y salen menos violaciones, con las mismas tres
comprobaciones.

**La M/G/1 y el factor (1 + CV²)/2.** Pollaczek–Khinchine da la espera media en cola de una
M/G/1:

> Wq = λ·E[S²] / (2(1 − ρ)) = ρ·E[S]/(1 − ρ) · (1 + CV²)/2

El primer factor es la M/M/1 de la misma media; el segundo, la **variabilidad del servicio**.
Con el triaje lognormal (CV ≈ 0,58) el factor vale 0,67: la espera es un tercio menor que con un
servicio exponencial de la misma media. Para verlo, la misma M/G/1 se simula también con un
triaje exponencial (`exponential_triage`): su espera sube a la de la M/M/1 y la comprobación
**falla**, como debe.

In [ ]:
SEMILLA = 2026
mg1, mg1_replicas = check_mg1(
    CENTRO, replicas=REPLICAS, days=DIAS_MG1, warmup_days=CAL, seed=SEMILLA
)
teoria = mg1_replicas.theory
cv2 = teoria.second_moment / teoria.mean_service**2 - 1
mm1 = teoria.rho * teoria.mean_service / (1 - teoria.rho)
print(
    f"λ = {60 * teoria.lam:g}/h, E[S] = {teoria.mean_service:.3f} min, CV² = {cv2:.3f},"
    f" (1 + CV²)/2 = {(1 + cv2) / 2:.3f}"
)
print(f"P–K {teoria.wq:.2f} min = M/M/1 {mm1:.2f} × {(1 + cv2) / 2:.3f}")
exponencial, exp_replicas = check_mg1(
    CENTRO,
    replicas=max(2, REPLICAS // 2),
    days=DIAS_MG1,
    warmup_days=CAL,
    seed=SEMILLA,
    sampler_factory=exponential_triage(mg1_center(CENTRO)),
)
print(
    f"Triaje lognormal: Wq simulada {mg1.statistic:.2f} [{mg1.ci_low:.2f}, {mg1.ci_high:.2f}]"
    f" → {'pasa' if mg1.passed else 'FALLA'}"
)
print(
    f"Triaje exponencial: Wq simulada {exponencial.statistic:.2f}"
    f" [{exponencial.ci_low:.2f}, {exponencial.ci_high:.2f}] frente a P–K {teoria.wq:.2f}"
    f" → {'pasa' if exponencial.passed else 'FALLA'} (debe fallar)"
)
display(plot_mg1(mg1_replicas, mg1.details[0]))

In [ ]:
mtginf, infinita, por_hora = check_infinite_server(
    CENTRO,
    PERFIL,
    replicas=REPLICAS,
    days=DIAS_MTGINF,
    warmup_days=CAL,
    seed=SEMILLA + 1,
    stay_seed=SEMILLA + 2,
    n_stays=ESTANCIAS,
)
leyes = check_operational_laws(CENTRO, PERFIL, days=DIAS_MES, warmup_days=CAL, seed=SEMILLA_MES)
print(format_engine_validation([*leyes, mg1, mtginf, infinita]))
display(plot_hourly_census(por_hora))

Es la misma tabla que imprime `edsim motor validar`, sin la comprobación de "más médicos", que
va en la sección 10. Las leyes operacionales usan la ejecución del mes de la sección 6.

**Para experimentar**

- Cambia la mediana del triaje (`dataclasses.replace(CENTRO, triage_time=(6.0, 10.0))`) en
  `check_mg1`. ¿Sigue pasando? ¿Por qué la teoría cambia con la configuración?
- ¿Con cuántas réplicas detecta `check_mg1` el triaje exponencial? (La tarea 3.6 vio que con 5
  no basta.)

> **Lo que debes recordar:** sin fórmula cerrada, el motor se valida con comprobaciones que miran cosas distintas: invariantes (forma), leyes operacionales (coherencia), M/G/1 y Mₜ/G/∞ (teoría) y condiciones extremas; la variabilidad del servicio, y no solo su media, produce espera: el factor (1 + CV²)/2.

## 10. Números aleatorios comunes

Con la misma semilla, **cada paciente es el mismo** aunque cambie la plantilla: llega al mismo
instante y con los mismos sorteos, porque todo lo aleatorio se sortea al llegar y con un flujo
por fuente. Solo cambian las esperas. Comparar dos plantillas **paciente a paciente** quita el
ruido de los pacientes: las esperas de un mismo paciente en las dos ejecuciones están
correlacionadas (llega en la misma hora punta, con la misma valoración larga), así que la
varianza de la diferencia es menor que la suma de las varianzas.

In [ ]:
mas = dataclasses.replace(CENTRO, doctors=tuple(d + 2 for d in CENTRO.doctors))
_, analisis_mas = ejecuta(mas, DIAS, SEMILLA)
_, analisis_base = ejecuta(CENTRO, DIAS, SEMILLA)
crn = paired_waits(analisis_base.table, analisis_mas.table, "doctor_wait", CAL * DIA, DIAS * DIA)
r = np.corrcoef(crn.base, crn.other)[0, 1]
print(f"{len(crn.episode_ids)} pacientes emparejados; correlación de sus esperas {r:.2f}")
print(
    f"Espera media de médico: {crn.base.mean():.2f} → {crn.other.mean():.2f} min"
    f" (diferencia {crn.difference.mean():+.2f})"
)
print(
    f"Varianza de la diferencia {crn.difference.var():.1f} frente a la suma de varianzas"
    f" {crn.base.var() + crn.other.var():.1f}"
)
print(
    f"Esperan menos {np.sum(crn.difference < 0)}, igual {np.sum(crn.difference == 0)}"
    f" y más {np.sum(crn.difference > 0)}"
)

fig, (ax_s, ax_h) = plt.subplots(1, 2, figsize=(12, 4.2))
ax_s.scatter(crn.base, crn.other, s=6, alpha=0.4, color=OBSERVED_COLOR)
tope = max(crn.base.max(), 1)
ax_s.plot([0, tope], [0, tope], color="k", ls=":", lw=1)
ax_s.set_xlabel("espera de médico, plantilla base (min)")
ax_s.set_ylabel("con 2 médicos más por turno (min)")
ax_s.set_title("Los mismos pacientes en dos ejecuciones")
ax_h.hist(crn.difference, bins=40, color=OBSERVED_COLOR, edgecolor="white")
ax_h.axvline(0, color="k", lw=1)
ax_h.set_yscale("log")
ax_h.set_xlabel("diferencia por paciente (min)")
ax_h.set_ylabel("pacientes (escala log)")
ax_h.set_title("La mayoría espera igual o menos; unos pocos, más")
plt.tight_layout()
plt.show()

In [ ]:
mas_medicos = check_more_doctors(
    CENTRO, PERFIL, replicas=PARES, days=DIAS_PAR, warmup_days=CAL, seed=SEMILLA
)
print(
    f"{PARES} pares de {DIAS_PAR} días: diferencia media {mas_medicos.statistic:+.2f} min,"
    f" IC al 99 % [{mas_medicos.ci_low:.2f}, {mas_medicos.ci_high:.2f}]"
    f" → {'pasa' if mas_medicos.passed else 'FALLA'}"
)

Con 10 pares el IC de la diferencia ya queda entero por debajo de 0: cada par comparte las
llegadas, el efecto diario Z y los sorteos, así que la variación entre semanas, mucho mayor que
el efecto de dos médicos, se cancela en la diferencia.

**Para experimentar**

- Compara la plantilla base con `mas` usando **semillas distintas** (`SEMILLA` y `SEMILLA + 1`):
  ¿qué pasa con la correlación y con la varianza de la diferencia?
- Algunos pacientes esperan **más** con más médicos. ¿Cómo puede ser? (Pista: con más médicos
  otros pacientes salen antes y cambia quién ocupa cada box y cuándo lo pide.)

> **Lo que debes recordar:** con la misma semilla cada paciente es el mismo en las dos ejecuciones y solo cambian sus esperas; emparejar paciente a paciente quita su ruido y permite ver efectos pequeños con pocas réplicas.

## 11. Para experimentar

Cambia los parámetros y ejecuta las dos celdas. El centro es `config/centro.toml` con los
cambios de abajo; la tabla es la misma que imprime `edsim motor simular`.

Preguntas guía:

- **Plantilla** (`MEDICOS_EXP = (11, 12, 7)` o `(12, 12, 9)`): ¿qué turno mejora más? ¿Y la
  espera de qué nivel?
- **Boxes** (`BOXES_EXP = 25` o `35`): ¿qué espera cambia, la de box o la de médico?
- **Duraciones** (`VALORACION_EXP = (25.0, 55.0)`): ¿cuánto sube la estancia? ¿Y las esperas?
- **Reparto de niveles** (`NIVELES_EXP = (5.0, 15.0, 30.0, 45.0, 5.0)`): ¿cuántos desalojos hay?
  ¿Quién paga la prioridad?
- **Semillas** (`SEMILLA_EXP`): ¿cuánto cambian las medias de una semana a otra?

In [ ]:
MEDICOS_EXP = CENTRO.doctors  # mañana, tarde, noche
BOXES_EXP = CENTRO.boxes
VALORACION_EXP = CENTRO.assessment_time  # (mediana, p90) en minutos
NIVELES_EXP = CENTRO.level_shares  # reparto del nivel 1 al 5
DIAS_EXP = DIAS
SEMILLA_EXP = 2026

centro_exp = dataclasses.replace(
    CENTRO,
    doctors=MEDICOS_EXP,
    boxes=BOXES_EXP,
    assessment_time=VALORACION_EXP,
    level_shares=NIVELES_EXP,
)
sink_exp = MemorySink()
res_exp = simulate_ed(centro_exp, PERFIL, DIAS_EXP, SEMILLA_EXP, sink=sink_exp, warmup_days=CAL)
analisis_exp = analyze_run(sink_exp.events, centro_exp, DIAS_EXP, CAL)
viol_exp = check_invariants(
    sink_exp.events, centro_exp, ground_truth=res_exp.ground_truth, inside=res_exp.inside
)
print(format_run_summary(analisis_exp, viol_exp))
print(f"Desalojos: {sum(e.event_type == 'patient_displaced' for e in sink_exp.events)}")

In [ ]:
display(plot_waits_by_level(analisis_exp.metrics))
display(plot_census(analisis_exp, centro_exp, PERFIL.start_date))

> **Lo que debes recordar:** cada cambio de la configuración se puede probar en segundos con la misma semilla; mira siempre las esperas por nivel y los días malos, no solo las medias.

## 12. Resumen

Lo que debes recordar de cada sección:

1. El motor es la M/M/1 con más piezas: λ(t), tres recursos retenidos en tramos distintos,
   prioridades y turnos.
2. El paciente es un proceso que espera en cada `yield` y emite un evento en cada transición;
   sus sorteos se hacen al llegar y solo sus esperas dependen de los demás.
3. El log es la única salida: censo, ocupación, esperas y "lo que se sabía" en cada instante
   salen de plegar sus eventos.
4. El box se retiene toda la estancia y el médico solo en las valoraciones: por eso los boxes se
   llenan antes.
5. La prioridad es la tupla `(nivel, llegada)`; el nivel 1 desaloja con reanudación, y la
   prioridad reparte la espera, no la reduce.
6. La carga ofrecida dice cuántos recursos ocupa la demanda sin colas; cerca de la plantilla, un
   médico de menos pesa mucho más que lejos de ella.
7. Con λ(t) hay un régimen periódico; el calentamiento se mide en días completos frente al mismo
   día de la semana siguiente.
8. Una ocupación media moderada esconde días saturados: se mira el porcentaje de días fuera de
   los máximos de Manchester.
9. Sin fórmula cerrada se valida con comprobaciones que miran cosas distintas; la variabilidad
   del servicio produce espera: (1 + CV²)/2.
10. Con números aleatorios comunes, los mismos pacientes en dos plantillas: el emparejamiento
    quita el ruido.

Las cuatro frases que pide el hito:

- **El box es el cuello de botella aunque haya médicos libres** porque el paciente lo retiene
  toda la estancia, también mientras espera las pruebas, y al médico solo durante las
  valoraciones.
- **La prioridad es una tupla** `(nivel, llegada)` para que el orden sea el del diseño aunque un
  paciente pida el recurso tarde (una reevaluación o la vuelta tras un desalojo).
- **El censo y la ocupación se derivan del log** sumando los cambios que produce cada evento en
  cada instante: el estado en t es el pliegue de los eventos hasta t.
- **Un sistema sin fórmula cerrada se valida** con invariantes, leyes operacionales, casos con
  teoría (M/G/1, Mₜ/G/∞), condiciones extremas y regresión: cada una ve errores que las otras no
  ven.